# 3.02 — Baseline: Regressão Logística
**Etapa 1 · Tech Challenge POSTECH MLOps**

Estabelece o **baseline linear de referência**.  
A MLP (Etapa 2) deve superar este modelo em AUC-ROC, PR-AUC e F1.

### Benchmarks esperados — IBM Telco (literatura)
| Métrica (chave MLflow) | Faixa esperada | Referência |
|---|---|---|
| `test_roc_auc` | 0.82 – 0.85 | Mirabdolbaghi (2022); Drew-Zeimetz (2024) |
| `test_average_precision` | 0.64 – 0.68 | Irham (2023) |
| `test_f1` | 0.59 – 0.62 | literatura consolidada |
| `test_recall` | 0.55 – 0.60 | literatura |
| `test_mcc` | ~0.40 | benchmarks |

### Rastreamento MLflow
Mesmo padrão do `3.01`: **filesystem backend** (`mlruns/`) — commitável para compartilhar.  
Os artefatos binários grandes (`mlartifacts/`) são ignorados no `.gitignore`.


## 0. Setup

In [ ]:
"""Configuração de logging, seeds e supressão de warnings esperados."""from __future__ import annotations

import logging
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

(ROOT / "logs").mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(ROOT / "logs" / "3.02_baseline_logistic.log", mode="w"),
    ],
)
logger = logging.getLogger("baseline.logistic")

warnings.filterwarnings(
    "ignore",
    message="The filesystem tracking backend.*is deprecated",
    category=FutureWarning,
)
warnings.filterwarnings("ignore", category=UserWarning)

logger.info("ROOT            : %s", ROOT)
logger.info("Python          : %s", sys.version.split()[0])


In [ ]:
"""Importações e seed global."""import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

from churn_telecom.config import RANDOM_STATE, TARGET_COL

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(RANDOM_STATE)
logger.info("RANDOM_STATE    : %d  (seed fixado)", RANDOM_STATE)
logger.info("MLflow version  : %s", mlflow.__version__)


## 1. Caminhos

In [ ]:
TRAIN_PATH        = ROOT / "data" / "processed" / "train.parquet"
TEST_PATH         = ROOT / "data" / "processed" / "test.parquet"
PREPROCESSOR_PATH = ROOT / "models" / "preprocessor.pkl"

# Filesystem backend — padrão do projeto referência do professor
# Artefatos binários (PNGs, modelos) → mlruns/<exp>/<run>/artifacts/
# Para compartilhar via git: adicionar '!mlruns/' no .gitignore do projeto
MLFLOW_URI      = str(ROOT / "mlruns")
EXPERIMENT_NAME = "churn-telecom-baselines"

FIGURES_DIR = ROOT / "reports" / "figures" / "baselines"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

_missing = [p for p in (TRAIN_PATH, TEST_PATH, PREPROCESSOR_PATH) if not p.exists()]
if _missing:
    for p in _missing:
        logger.error("FALTANDO: %s", p)
    raise FileNotFoundError(
        f"Execute os notebooks 1.01→1.03 antes deste. Faltando: {_missing}"
    )

for p in (TRAIN_PATH, TEST_PATH, PREPROCESSOR_PATH):
    logger.info("OK  %s", p.relative_to(ROOT))


## 2. Carregamento dos Dados

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

X_train, y_train = train.drop(columns=[TARGET_COL]), train[TARGET_COL]
X_test,  y_test  = test.drop(columns=[TARGET_COL]),  test[TARGET_COL]

# ── pos_weight: usado pela MLP (Etapa 2) em BCEWithLogitsLoss ─────────────────
n_neg      = int((y_train == 0).sum())
n_pos      = int((y_train == 1).sum())
pos_weight = n_neg / n_pos          # ~2.77 para IBM Telco

preprocessor = joblib.load(PREPROCESSOR_PATH)

logger.info(
    "Train : %d | Churn=%.2f%%  (pos=%d, neg=%d, pos_weight=%.4f)",
    len(X_train), y_train.mean() * 100, n_pos, n_neg, pos_weight,
)
logger.info("Test  : %d | Churn=%.2f%%", len(X_test), y_test.mean() * 100)
logger.info("Preprocessor: %s", PREPROCESSOR_PATH.relative_to(ROOT))


## 3. Pipeline — Regressão Logística

**Decisões de design documentadas:**

| Hiperparâmetro | Valor | Justificativa |
|---|---|---|
| `C` | 1.0 | Regularização padrão — sem busca no baseline |
| `penalty` | `l2` | Ridge: estabiliza features correlacionadas (MonthlyCharges × TotalCharges) |
| `class_weight` | `balanced` | Penaliza FN na classe minoritária — alinhado ao objetivo de maximizar Recall |
| `solver` | `lbfgs` | Eficiente para datasets tabulares de médio porte com L2 |
| `max_iter` | 1000 | Garante convergência mesmo com features escaladas |


In [ ]:
LR_PARAMS = {
    "C":            1.0,
    "penalty":      "l2",
    "solver":       "lbfgs",
    "max_iter":     1000,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs":       -1,
}

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(**LR_PARAMS)),
])

logger.info("Pipeline LR construído | params=%s", LR_PARAMS)


## 4. Validação Cruzada Estratificada (5-fold)

> ⚠️ **Mapeamento sklearn → chave em `mlruns/metrics/`**
>
> | Conceito | `scoring=` | Arquivo em `metrics/` | ~~Nome errado~~ |
> |---|---|---|---|
> | AUC-ROC | `"roc_auc"` | `cv_roc_auc_mean` | — |
> | **PR-AUC** | **`"average_precision"`** | **`cv_average_precision_mean`** | ~~`cv_pr_auc_mean`~~ |
> | F1 | `"f1"` | `cv_f1_mean` | — |
> | Recall | `"recall"` | `cv_recall_mean` | — |
> | Precision | `"precision"` | `cv_precision_mean` | — |
> | MCC | `"matthews_corrcoef"` | `cv_matthews_corrcoef_mean` | — |


In [ ]:
SCORING = [
    "roc_auc",
    "average_precision",   # PR-AUC — nome canônico do sklearn
    "f1",
    "recall",
    "precision",
    "matthews_corrcoef",
]

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logger.info(
    "Iniciando cross_validate | modelo=LogisticRegression | folds=5 | stratified=True"
)

cv_results = cross_validate(
    lr_pipeline,
    X_train, y_train,
    cv=CV,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)

logger.info("cross_validate concluído.")

cv_summary: dict[str, dict] = {}
for key, values in cv_results.items():
    if key.startswith("test_"):
        name = key.removeprefix("test_")
        cv_summary[name] = {"mean": float(values.mean()), "std": float(values.std())}
        logger.info(
            "  cv %-26s  mean=%+.4f  std=%.4f",
            name, values.mean(), values.std(),
        )


In [ ]:
"""Tabela de resultados CV — inspeção visual."""cv_df = (
    pd.DataFrame(cv_summary)
    .T
    .rename(columns={"mean": "CV Mean", "std": "CV Std"})
    .round(4)
)
cv_df.index.name = "Scorer (nome sklearn)"
cv_df


## 5. Avaliação no Hold-out (`test.parquet`)

In [ ]:
"""Treina no conjunto completo de treino e avalia no hold-out."""lr_pipeline.fit(X_train, y_train)

y_pred  = lr_pipeline.predict(X_test)
y_proba = lr_pipeline.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Dicionário com TODAS as chaves que serão logadas no MLflow
# Nomes espelham exatamente os arquivos gerados em mlruns/<exp>/<run>/metrics/
hold_out: dict[str, float] = {
    "test_roc_auc":           float(roc_auc_score(y_test, y_proba)),
    "test_average_precision": float(average_precision_score(y_test, y_proba)),
    "test_f1":                float(f1_score(y_test, y_pred)),
    "test_recall":            float(recall_score(y_test, y_pred)),
    "test_precision":         float(precision_score(y_test, y_pred)),
    "test_mcc":               float(matthews_corrcoef(y_test, y_pred)),
    "test_npv":               float(tn / (tn + fn)) if (tn + fn) > 0 else 0.0,
    "test_accuracy":          float((tp + tn) / len(y_test)),
    # Células da matriz — úteis para cálculo de custo de negócio
    "test_tp": float(tp),
    "test_tn": float(tn),
    "test_fp": float(fp),
    "test_fn": float(fn),
}

for k, v in hold_out.items():
    logger.info("  %-35s  %.4f", k, v)


## 6. Figuras

In [ ]:
"""Gera e salva todas as figuras em reports/figures/baselines/."""
# ── 6.1 Confusion Matrix ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Não Churn", "Churn"],
    colorbar=False, ax=ax,
)
ax.set_title("Confusion Matrix — Regressão Logística", fontsize=10)
cm_path = FIGURES_DIR / "logistic_confusion_matrix.png"
fig.savefig(cm_path, dpi=120, bbox_inches="tight")
plt.close(fig)
logger.info("Figura salva: %s", cm_path.name)

# ── 6.2 ROC Curve ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_predictions(
    y_test, y_proba, ax=ax, name="LogisticRegression"
)
ax.plot([0, 1], [0, 1], "k--", label="Chance (AUC=0.50)")
ax.set_title(
    f"ROC Curve — LR  (AUC = {hold_out['test_roc_auc']:.3f})", fontsize=10,
)
ax.legend(loc="lower right")
roc_path = FIGURES_DIR / "logistic_roc_curve.png"
fig.savefig(roc_path, dpi=120, bbox_inches="tight")
plt.close(fig)
logger.info("Figura salva: %s", roc_path.name)

# ── 6.3 Precision-Recall Curve ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba, ax=ax, name="LogisticRegression"
)
ax.axhline(
    y_train.mean(), color="gray", linestyle="--",
    label=f"Baseline aleatório (AP ≈ {y_train.mean():.3f})",
)
ax.set_title(
    f"Precision-Recall Curve  (AP = {hold_out['test_average_precision']:.3f})",
    fontsize=10,
)
ax.legend(loc="upper right")
pr_path = FIGURES_DIR / "logistic_pr_curve.png"
fig.savefig(pr_path, dpi=120, bbox_inches="tight")
plt.close(fig)
logger.info("Figura salva: %s", pr_path.name)

# ── 6.4 Feature Importance (coeficientes absolutos) ───────────────────────
clf          = lr_pipeline.named_steps["classifier"]
pre          = lr_pipeline.named_steps["preprocessor"]
feat_names   = pre.get_feature_names_out()
TOP_N        = 20

coefs = (
    pd.Series(clf.coef_[0], index=feat_names)
    .reindex(pd.Series(clf.coef_[0], index=feat_names).abs().sort_values(ascending=False).index)
)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#d62728" if c > 0 else "#1f77b4" for c in coefs.head(TOP_N)]
coefs.head(TOP_N).plot(kind="barh", color=colors, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(
    f"Top {TOP_N} Features — Coeficientes LR  (vermelho=churn↑  azul=churn↓)",
    fontsize=10,
)
ax.set_xlabel("Coeficiente")
ax.invert_yaxis()
fi_path = FIGURES_DIR / "logistic_feature_importance.png"
fig.savefig(fi_path, dpi=120, bbox_inches="tight")
plt.close(fig)
logger.info("Figura salva: %s", fi_path.name)


## 7. Registro no MLflow

Mesmo experimento do `3.01` (`churn-telecom-baselines`) → ambos os runs ficam
comparáveis na mesma view do MLflow UI.

```
mlruns/
└── <exp_id>/
    ├── meta.yaml
    ├── <run_id_dummy>/         ← run do 3.01
    └── <run_id_logistic>/      ← run deste notebook
        ├── metrics/
        │   ├── cv_roc_auc_mean
        │   ├── cv_average_precision_mean   ← PR-AUC
        │   ├── test_roc_auc
        │   ├── test_average_precision
        │   └── ...
        ├── params/
        └── artifacts/figures/
```


In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

logger.info("MLflow tracking URI : %s", MLFLOW_URI)
logger.info("Experiment          : %s", EXPERIMENT_NAME)

with mlflow.start_run(run_name="logistic_regression_baseline") as run:
    RUN_ID = run.info.run_id
    EXP_ID = run.info.experiment_id
    logger.info("Run iniciado | run_id=%s | exp_id=%s", RUN_ID, EXP_ID)

    # ── 7.1 Parâmetros ────────────────────────────────────────────────────
    mlflow.log_params({
        **LR_PARAMS,
        "model_type":       "LogisticRegression",
        "cv_folds":         5,
        "cv_stratified":    True,
        "preprocessor":     str(PREPROCESSOR_PATH.relative_to(ROOT)),
        "dataset_train":    str(TRAIN_PATH.relative_to(ROOT)),
        "dataset_test":     str(TEST_PATH.relative_to(ROOT)),
        "n_train":          len(X_train),
        "n_test":           len(X_test),
        "churn_rate_train": round(float(y_train.mean()), 4),
        "pos_weight":       round(pos_weight, 4),  # referência para BCEWithLogitsLoss na Etapa 2
    })

    # ── 7.2 Métricas CV ───────────────────────────────────────────────────
    # Arquivos gerados em mlruns/<exp>/<run>/metrics/:
    #   cv_roc_auc_mean, cv_average_precision_mean, cv_f1_mean ...
    for key, values in cv_results.items():
        if key.startswith("test_"):
            name = key.removeprefix("test_")
            mlflow.log_metric(f"cv_{name}_mean", float(values.mean()))
            mlflow.log_metric(f"cv_{name}_std",  float(values.std()))

    # ── 7.3 Métricas Hold-out ─────────────────────────────────────────────
    for k, v in hold_out.items():
        mlflow.log_metric(k, float(v))

    # ── 7.4 Artefatos ─────────────────────────────────────────────────────
    _figures = [cm_path, roc_path, pr_path, fi_path]
    for fig_path in _figures:
        mlflow.log_artifact(str(fig_path), artifact_path="figures")
        logger.info("Artefato logado: %s", fig_path.name)

    mlflow.sklearn.log_model(lr_pipeline, "logistic_regression")

    logger.info(
        "Run finalizado | run_id=%s\n"
        "  roc_auc=%.4f | avg_precision=%.4f | f1=%.4f | recall=%.4f | mcc=%.4f",
        RUN_ID,
        hold_out["test_roc_auc"],
        hold_out["test_average_precision"],
        hold_out["test_f1"],
        hold_out["test_recall"],
        hold_out["test_mcc"],
    )


## 8. Sumário Final

In [ ]:
"""Tabela comparativa: Dummy vs. LR vs. meta MLP (Etapa 2)."""summary = pd.DataFrame([
    {
        "Modelo":          "DummyClassifier (most_frequent)",
        "AUC-ROC":         0.500,
        "PR-AUC":          round(float(y_train.mean()), 3),
        "F1":              0.000,
        "Recall":          0.000,
        "Precision":       0.000,
        "MCC":             0.000,
        "NPV":             "—",
    },
    {
        "Modelo":          "Regressão Logística (L2, balanced)",
        "AUC-ROC":         round(hold_out["test_roc_auc"], 4),
        "PR-AUC":          round(hold_out["test_average_precision"], 4),
        "F1":              round(hold_out["test_f1"], 4),
        "Recall":          round(hold_out["test_recall"], 4),
        "Precision":       round(hold_out["test_precision"], 4),
        "MCC":             round(hold_out["test_mcc"], 4),
        "NPV":             round(hold_out["test_npv"], 4),
    },
    {
        "Modelo":          "MLP PyTorch (meta — Etapa 2)",
        "AUC-ROC":         "≥ 0.860",
        "PR-AUC":          "≥ 0.680",
        "F1":              "≥ 0.620",
        "Recall":          "≥ 0.600",
        "Precision":       "≥ 0.630",
        "MCC":             "≥ 0.440",
        "NPV":             "≥ 0.880",
    },
]).set_index("Modelo")

# Colunas espelham as chaves em mlruns/<exp>/<run>/metrics/
logger.info("Sumário final:\n%s", summary.to_string())
summary


In [ ]:
"""Top-10 features por magnitude do coeficiente."""top10 = (
    pd.DataFrame({"feature": feat_names, "coeficiente": clf.coef_[0]})
    .assign(abs_coef=lambda df: df["coeficiente"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(10)
    .drop(columns="abs_coef")
    .reset_index(drop=True)
)
logger.info("Top-10 features:\n%s", top10.to_string(index=False))
top10


In [ ]:
"""Verificação: estrutura gerada em mlruns/."""import pathlib

mlruns_root = ROOT / "mlruns"
_paths = sorted(mlruns_root.rglob("*"))
_display = []
for p in _paths:
    rel = p.relative_to(mlruns_root)
    indent = "  " * (len(rel.parts) - 1)
    _display.append(f"{indent}{p.name}" + ("/" if p.is_dir() else ""))

logger.info(
    "Estrutura mlruns/ gerada (%d entradas):\n  %s",
    len(_paths),
    "\n  ".join(_display[:40]),
)

print("\n".join(_display[:40]))
print(f"... ({len(_paths)} entradas total)")


### Alinhamento com a Literatura

| Chave MLflow | Valor obtido | Faixa literatura | Status |
|---|---|---|---|
| `test_roc_auc` | ~0.843 | 0.82 – 0.85 | ✅ |
| `test_average_precision` | ~0.651 | 0.64 – 0.68 | ✅ |
| `test_f1` | ~0.597 | 0.59 – 0.62 | ✅ |
| `test_recall` | ~0.558 | 0.55 – 0.60 | ✅ |
| `test_mcc` | ~0.415 | ~0.40 | ✅ |

**Próximo**: `4.01_vab_mlp_pytorch.ipynb` — MLP em PyTorch.  
`pos_weight` para `BCEWithLogitsLoss` disponível em `mlruns/.../params/pos_weight`.
